# RQS 2026 Summer School — Problem Set

**Instructor:** Prof. Yongshan Ding, **Problem set designed by:** Dantong Li

*Built on PennyLane -- Jordan–Wigner routing on 9 fermionic modes / a 3×3 qubit grid*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yale-QCS/rqs2026-pset/blob/main/problem_set.ipynb)

**What you'll do.** 

1. **Problem 1** — why "swap two qubits" $\neq$ "swap two fermions", and the
   one gate that fixes it.
2. **Problem 2** — reach a *distant* mode using only adjacent swaps (a bubble walk).
3. **Problem 3** — a three-pair permutation, scheduled for minimum depth.
4. **Problem 4** — one Trotter step of 2-D Fermi–Hubbard on a nearest-neighbour grid.
5. **Problem 5** — the same step on all-to-all hardware.

Every tool lives in the `pset_tools` package beside this notebook.


## Setup

Run the two cells below once:

* **Google Colab** — installs PennyLane and clones [`Yale-QCS/rqs2026-pset`](https://github.com/Yale-QCS/rqs2026-pset)
  so the `pset_tools` toolkit (and its readable source) lands in the file pane
  on the left. Takes ~1 minute the first time.
* **Local Jupyter** — if you cloned the repo, everything is already next to
  this notebook and the cell is a no-op. Dependencies, if you need them:
  `pip install pennylane matplotlib numpy ipywidgets`.


In [ ]:
# ▶ Bootstrap — make dependencies + the pset_tools package available.
import importlib.util, os, subprocess, sys

REPO_URL = "https://github.com/Yale-QCS/rqs2026-pset"
REPO_DIR = "rqs2026-pset"

# 1) PennyLane (the only dependency Colab doesn't ship with).
if importlib.util.find_spec("pennylane") is None:
    print("Installing PennyLane (~1 min)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pennylane"],
                   check=True)

# 2) pset_tools: use a local copy if it sits next to this notebook (local
#    checkout, or an uploaded/unzipped folder), otherwise clone the repo
#    (the Colab path).
if importlib.util.find_spec("pset_tools") is None:
    _here = os.getcwd()
    _cands = [_here, os.path.join(_here, REPO_DIR)]
    _target = next((c for c in _cands
                    if os.path.isdir(os.path.join(c, "pset_tools"))), None)
    if _target is None:
        print(f"Cloning {REPO_URL} ...")
        _r = subprocess.run(["git", "clone", "-q", "--depth", "1",
                             REPO_URL, REPO_DIR])
        if _r.returncode != 0:
            raise RuntimeError(
                f"Could not clone {REPO_URL} (is the repo public yet?).\n"
                f"Fallback: upload `{REPO_DIR}.zip` via the file pane, run\n"
                f"    !unzip -q {REPO_DIR}.zip\n"
                f"in a new cell, then re-run this cell."
            )
        _target = os.path.join(_here, REPO_DIR)
    if _target not in sys.path:
        sys.path.insert(0, _target)

# 3) Colab: switch on the richer widget manager (for the slider scrubber).
if "google.colab" in sys.modules:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()

print("Bootstrap done.")


In [ ]:
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt

from pset_tools import (
    # checkers
    verify_fp, verify_trotter_step,
    # depth report (validates the native gate set + topology, then counts
    # two-qubit depth).  There is no compiler: you build every component yourself
    # -- including the fermionic swap, which you implement in Problem 1.
    depth_2q,
    # visualizers
    draw_majorana_tree, interactive_majorana_tree, animate_majorana_tree,
    play_majorana_tree, majorana_strings_after,
    snake_layout_figure, fermion_lattice_figure, snake_and_lattice_figure,
    permutation_on_snake_figure,
    # helpers used in Problem 4
    HORIZONTAL_EDGES, VERTICAL_EDGES, HOPPING_EDGES,
    hopping_pauli_strings,
    # for reference if you want to inspect the JW form by hand
    initial_majorana, snake_to_rc, N_MODES,
)

print(f"pset_tools loaded.  N_MODES = {N_MODES}.  PennyLane {qml.version()}.")


## Warm-Up

*Introduction to Fermionic simulations.*

As shown in lecture, a second quantized Fermionic hamiltonian has the form:

$$
H =  \sum_{j,k} h_{jk} a_j^\dagger a_k + \frac{1}{2} \sum_{j,k,l,m} h_{jklm} a_j^\dagger a_k^\dagger a_l a_m
$$

In this exercise, we consider 9 fermionic modes labelled $0, 1, \ldots, 8$, each with creation and
annihilation operators $a_j^\dagger, a_j$ obeying the canonical
anticommutation relations

$$
\{a_j, a_k^\dagger\} = \delta_{jk},
\qquad
\{a_j, a_k\} = 0,
\qquad
\{a_j^\dagger, a_k^\dagger\} = 0.
$$

### Majorana operators

It is easier to reason about fermionic permutations if we split each mode
into two **Majorana operators**:

$$
\gamma_{2j} = a_j^\dagger + a_j,
\qquad
\gamma_{2j+1} = i\,(a_j^\dagger - a_j).
$$

These are the real and imaginary parts of $a_j^\dagger$ (up to factors of
$i$).  They are Hermitian and obey the clean algebra

$$
\gamma_p^2 = I, \qquad \{\gamma_p, \gamma_q\} = 0 \text{ for } p \neq q.
$$

So the $2N = 18$ Majoranas are a set of mutually anticommuting Hermitian
operators that square to the identity — much simpler than the $a_j,
a_j^\dagger$.  A fermionic permutation that sends mode $j$ to mode $\pi(j)$
must send the *ordered pair* $(\gamma_{2j}, \gamma_{2j+1})$ to
$(\gamma_{2\pi(j)}, \gamma_{2\pi(j)+1})$ — no sign changes, no mixing within
the pair.

### The Jordan–Wigner mapping

Under the **Jordan–Wigner (JW) mapping**, each mode lives on one qubit and the
Majoranas become Pauli strings:

$$
\gamma_{2j} = Z_0 Z_1 \cdots Z_{j-1} X_j,
\qquad
\gamma_{2j+1} = Z_0 Z_1 \cdots Z_{j-1} Y_j.
$$

The Pauli at mode $j$ is an $X$ or $Y$; everything to its left is a $Z$, and
everything to its right is identity (omitted).  Those trailing $Z$'s are the
**JW parity string** — the qubit-level mechanism that makes the $\gamma_p$'s
anticommute across different modes.

> ⚠️ **Heads-up**
>
> This parity string is why **swapping two qubits is not the same as swapping two
> fermions**. Mode $j$'s Majoranas have Pauli weight $j+1$. A qubit swap only
> relabels *which qubit holds which Pauli* — it cannot change how many qubits a
> Majorana acts on. So no qubit-swap circuit can turn $\gamma_6$ (weight 4) into
> $\gamma_8$ (weight 5). **Problem 1 makes this concrete.**

### Snake JW layout

We place the 9 qubits on a 3×3 grid in the **snake JW layout** — even rows
left-to-right, odd rows right-to-left.  Qubit $q_i$ initially holds mode $i$.
Two qubits are **JW-adjacent** iff their indices differ by 1, e.g. $(q_2,
q_3)$ are JW-adjacent even though they sit on different rows.


In [ ]:
fig, ax = snake_layout_figure(figsize=(5.4, 5.4))
plt.show()


### What "implementing a fermionic permutation" means

Intuitively, a **fermionic permutation $\pi$** rearranges which mode lives on
which qubit.  Precisely, we require conjugation by your unitary $U$ to permute
the Majoranas in ordered pairs:

> 🔑 **The rule we check**
>
> $$U\,\gamma_{2j+\alpha}\,U^\dagger \;=\; \gamma_{2\pi(j)+\alpha}
> \qquad \forall\, j \in \{0,\dots,8\},\ \alpha \in \{0,1\}.$$
>
> Each $\gamma_p$ is a specific Pauli string, so this is checked by watching how
> the 18 strings evolve through your circuit — exactly what `verify_fp` tests and
> `draw_majorana_tree` lets you see.

> 🧰 **Tools**
>
> - **`verify_fp(circuit, target_perm)`** — returns `True`/`False`. Pass; you're done. Fail; read on.
> - **`draw_majorana_tree(circuit, step=…)`** — the **morphing** ternary tree after the first `step` gates. The mapping is read root→leaf (left = $X$, middle = $Y$, right = $Z$), and all three parts of the tree are free to move: a physical `SWAP` exchanges two **qubit-node** labels, your `fswap` exchanges two **γ-leaf** labels, and `CZ`/`CNOT`/`CY` are **tree rotations** that change the **shape**. The tree is rebuilt from the operators themselves, so the picture is correct no matter how a gate is decomposed. (Pass `mode="strings"` for the older view that keeps the tree fixed and prints each Majorana's Pauli string instead; the tree view falls back to it automatically if a 2-qubit gate lands off a tree edge, which has no ternary-tree form.)
> - **`interactive_majorana_tree(circuit)`** — an ipywidgets slider to step through gate-by-gate; **`animate_majorana_tree`** plays it in place; and **`play_majorana_tree(circuit)`** returns a self-contained, continuously-tweened HTML animation (nodes and γ-leaves glide to their new spots).
> - **`depth_2q(circuit, connectivity=None)`** — checks your circuit is in the native gate set (and, if `connectivity="2dnn"`, that every two-qubit gate is grid-adjacent), then returns the **two-qubit depth**: the longest chain of two-qubit gates that pairwise share a qubit (single-qubit gates are free). There is **no compiler**, but a gate is accepted whenever it *decomposes* to native gates — so `qml.SWAP` counts as its 3 CNOTs, `qml.PauliRot` as its CNOT staircase, and your Problem-1 `fswap` as whatever primitives you built it from.
> - **Circuit shape.** Every tool above takes your circuit as a list of gates — and the list **may be nested**, so you can group sub-circuits as *gadgets*, e.g. `[fswap(2), fswap(3)]` or `[layer_a, layer_b]`. Nested lists are flattened in the order you wrote them.

Try them now — here is the tree before any gate is applied:


In [ ]:
# Initial JW Majorana strings (before any gate is applied).
fig, ax = draw_majorana_tree()
plt.show()


### How each gate moves the tree

*A mapping is three things — shape, internal node labels (qubits), leaf labels (Majoranas γ).*

Read the tree root→leaf: taking a node's **left ($X$, red)**, **middle ($Y$,
green)**, or **right ($Z$, blue)** child writes that Pauli on that qubit.  So the
mapping is exactly three pieces of data, all free to change — the tree **shape**,
the **qubit label** on each node, and the **Majorana ($\gamma$) label** on each
leaf.  A gate acts by conjugation $\gamma_p \mapsto U\,\gamma_p\,U^\dagger$, and
on the tree that becomes one tidy move:

| gate | move on the tree |
| :-- | :-- |
| single-qubit Clifford on qubit $q$ | permute the three child-edges at node $q$ — $H$ swaps **X↔Z**, $S$ swaps **X↔Y**, a Pauli flips $\pm$ signs |
| `SWAP(a, b)` | swap two **qubit-node** labels (shape & $\gamma$-leaves stay) |
| fermionic swap *(you build it in Problem 1)* | swap two **Majorana ($\gamma$) leaf-pairs** (shape & qubits stay) |
| `CZ` / `CNOT` / `CY` on a tree edge | a **tree rotation** — the *shape* changes (`CZ` transposes two spine nodes; `CNOT` rotates the $X/Z$ subtree; `CY` the $Y/Z$ subtree) |
| any 2-qubit gate *off* a tree edge | not a ternary tree at all → the view falls back to printing Pauli strings |

Three of these are just the **three ways to exchange two neighbouring nodes**:
`SWAP` repaints the *qubit* labels, the fermionic swap repaints the *Majorana*
labels, and `CZ` does **both at once** — which is exactly why `CZ` reads as a
rotation.  (Signs are a harmless gauge: flipping a $\gamma$'s sign is a depth-1
single-qubit Pauli.)


In [ ]:
# (1) A single-qubit Clifford touches ONE node — it permutes that node's three
#     coloured child-edges.  Compare each to the plain JW tree above.
draw_majorana_tree([qml.Hadamard(3)],
                   title="Hadamard on q3 — swaps the X (red) and Z (blue) children")
plt.show()
draw_majorana_tree([qml.S(3)],
                   title="S on q3 — swaps the X (red) and Y (green) children")
plt.show()


In [ ]:
# (2) SWAP keeps the SHAPE and swaps two QUBIT labels.  (Problem 1's fermionic
#     swap instead keeps shape+qubits and swaps two gamma-leaf labels.)
draw_majorana_tree([qml.SWAP(wires=[3, 4])],
                   title="SWAP(q3, q4) — the two qubit labels swap; gamma-leaves stay")
plt.show()


In [ ]:
# (3) The entangling Cliffords are tree ROTATIONS — they change the SHAPE.
draw_majorana_tree([qml.CZ(wires=[3, 4])],
                   title="CZ(3,4) — rotation: q3 and q4 transpose on the spine")
plt.show()
draw_majorana_tree([qml.CNOT(wires=[3, 4])],
                   title="CNOT(3,4) — rotation of the X/Z subtree")
plt.show()
draw_majorana_tree([qml.CY(wires=[3, 4])],
                   title="CY(3,4) — rotation of the Y/Z subtree")
plt.show()


### Watch a Clifford morph the tree

Stack several moves and the tree morphs continuously.
`play_majorana_tree(circuit)` returns a self-contained animation — the qubit
nodes and $\gamma$-leaves glide to their new spots.  Here is a Clifford using
**one of every move**, kept on tree edges so the mapping stays a tree throughout:


In [ ]:
clifford_demo = [
    qml.CZ(wires=[0, 1]),     # rotation
    qml.CNOT(wires=[2, 3]),   # rotation (X/Z subtree)
    qml.CY(wires=[4, 5]),     # rotation (Y/Z subtree)
    qml.Hadamard(6),          # recolour node 6 (X<->Z)
    qml.S(7),                 # recolour node 7 (X<->Y)
    qml.SWAP(wires=[0, 8]),   # swap two qubit labels
]
play_majorana_tree(clifford_demo)


And a second, busier one — a parallel layer of rotations, then a relabel:


In [ ]:
clifford_demo_2 = [
    qml.CNOT(wires=[1, 0]), qml.CY(wires=[3, 2]),
    qml.CZ(wires=[5, 4]),   qml.CNOT(wires=[7, 6]),
    qml.Hadamard(8),        qml.SWAP(wires=[0, 6]),
]
play_majorana_tree(clifford_demo_2)


---

## Problem 1 — Swapping two adjacent fermions

*Start with the smallest case: one adjacent transposition.*

> 🎯 **Task**
>
> Implement the **adjacent fermionic swap** as a reusable function `fswap(j)` that
> returns a list of native gates swapping modes $j$ and $j{+}1$. Test it on modes
> 3, 4 with `verify_fp(fswap(3), (3, 4))`.  **You will reuse `fswap` in every later
> problem**, so make it cheap.

> 💡 **Hint**
>
> First try a plain qubit `SWAP` on $q_3, q_4$ — the checker returns `False`. Open
> the Majorana visualizer and compare what your circuit produces against what
> $\gamma_6, \gamma_7, \gamma_8, \gamma_9$ *should* be. Find the discrepancy on the
> boundary between modes 3 and 4, then add one corrective two-qubit gate.

> 📦 **Deliverables**
>
> Your `fswap(j)`, its checker output on $(3,4)$, and its `depth_2q`.


### Step 1 — Does a plain qubit SWAP work?

Confirm the checker reports `False`, and use the visualizer to see *why*: by the
dictionary above, a bare `SWAP` only **swaps the two qubit labels** — but a
fermionic swap of modes 3 and 4 must instead swap the two **$\gamma$-leaf-pairs**
($\gamma_6,\gamma_7 \leftrightarrow \gamma_8,\gamma_9$).


In [ ]:
buggy_circuit_p1 = [qml.SWAP(wires=[3, 4])]
print("verify_fp:", verify_fp(buggy_circuit_p1, (3, 4)))
print("\n--- diagnostics on each Majorana ---")
verify_fp(buggy_circuit_p1, (3, 4), verbose=True)


In [ ]:
# Inspect the resulting Majorana strings visually.
fig, ax = draw_majorana_tree(buggy_circuit_p1,
                             title="After bare SWAP(q3, q4)")
plt.show()


### Step 2 — your fix, packaged as a reusable gate

Add one corrective two-qubit gate on $(q_3, q_4)$ so the parity strings of
$\gamma_6, \gamma_7$ gain the missing $Z_3$ and $\gamma_8, \gamma_9$ lose the
spurious $Z_4$.  Then wrap the construction into a function `fswap(j)` acting on
$(j, j{+}1)$, so the rest of the pset can call it.


In [ ]:
# YOUR CODE HERE — make this a CORRECT adjacent fermionic swap on (j, j+1).
def fswap(j):
    return [qml.SWAP(wires=[j, j + 1])]      # a bare SWAP is NOT enough -- fix it

my_fswap_3 = fswap(3)
print("verify_fp:", verify_fp(my_fswap_3, (3, 4)))   # False until fswap is correct
print("depth_2q :", depth_2q(my_fswap_3))
fig, ax = draw_majorana_tree(my_fswap_3, title="After your fswap(3)")
plt.show()


### Two ways to build it — and why the depth differs

There are two natural correct constructions, and **they are the same unitary**
(a `SWAP` with a $-1$ phase on $\lvert 11\rangle$):

| construction | native two-qubit gates | `depth_2q` |
|---|---|---|
| `SWAP(j, j+1)` then a corrective `CZ(j, j+1)` | `SWAP` (= 3 CNOTs) + `CZ` | **4** |
| `H · CNOT · CNOT · H` (the textbook fermionic SWAP) | 2 CNOTs | **2** |

`depth_2q` doesn't know what an "fswap" *is* — it just decomposes whatever ops
you hand it down to native gates (single-qubit + CNOT/CZ) and counts.  So the
two-CNOT form is strictly cheaper.  From here on, `fswap(j)` can be used for Problems 2–5.


---

## Problem 2 — Swapping two distant fermions

*You have one adjacent swap. Now reach across the chain using only adjacent swaps.*

> *Decouple the gate from the schedule: assume your `fswap` works, and treat fermionic permutation as a routing problem.*

> 🎯 **Task**
>
> Swap modes 2 and 6.  They sit at JW positions 2 and 6 — **JW distance 4** along
> the snake.  Use only your adjacent fermionic swap `fswap(j)` from Problem 1, and
> verify with `verify_fp(circuit, (2, 6))`.

> 💡 **Hint**
>
> This is the classic problem of swapping two distant elements of a list when you
> may only swap *adjacent* elements — think bubble sort.

> 📦 **Deliverables**
>
> Your circuit.


In [ ]:
# YOUR CODE HERE.  Concatenate fswap(j) lists with `+`, e.g.
#   my_swap_2_6 = fswap(2) + fswap(3) + ...
my_swap_2_6 = []

print("verify_fp:", verify_fp(my_swap_2_6, (2, 6)))
print("depth_2q :", depth_2q(my_swap_2_6))


Scrub through your circuit to watch modes 2 and 6 swap.  The animation redraws
the Majorana tree **in place**, one gate at a time (~1 s/frame); frame `k` is
the state after the first `k` gates — the same `k` you'd pass to
`draw_majorana_tree(my_swap_2_6, step=k)`.

> ⚠️ **Heads-up**
>
> ▶ **This is an animation — you must _run_ the cell to watch it play.** Each
> frame has an orange border to set it apart from the still figures. A
> saved/exported notebook keeps only the *last* frame (a live kernel is needed to
> replay the loop), so re-run the cell whenever you want to see it again.


In [ ]:
# Plays the steps in place (clears + redraws each frame, ~1s each).
# Run this cell to watch it; it needs a live kernel.  For a draggable
# slider instead, use interactive_majorana_tree(my_swap_2_6).
animate_majorana_tree(my_swap_2_6, pause=1.0)


---

## Problem 3 — 2-D reflection across the diagonal

*A permutation pattern commonly seen in Fermionic Fourier Transforms.*

Picture the nine modes on the zig-zag-labelled 3×3 grid (the snake layout from the
Background).  Reflecting this grid across its **main diagonal** (swapping rows
with columns) fixes the diagonal modes $0, 4, 8$ and exchanges the three
off-diagonal pairs:

$$
\pi = (1\;5)\,(2\;6)\,(3\;7),
\qquad\text{i.e. swap the modes at JW positions }
1\leftrightarrow5,\; 2\leftrightarrow6,\; 3\leftrightarrow7.
$$

Run the cell below to see the three swaps on the snake:


In [ ]:
fig, ax = permutation_on_snake_figure()   # pairs (1,5), (2,6), (3,7)
plt.show()


> *You'll meet this exact reflection again in Problem 4 — there it turns every expensive vertical hop into a cheap JW-adjacent one.*

> 🎯 **Task**
>
> Build a circuit, using only your adjacent fermionic swaps `fswap(j)`, that
> implements $\pi = (1\;5)(2\;6)(3\;7)$. Verify with
> `verify_fp(circuit, [(1,5),(2,6),(3,7)])`, then **minimize `depth_2q`**.

> 💡 **Hint**
>
> An `fswap(j)` and an `fswap(k)` on **disjoint** qubits run in the **same
> layers**; two that share a qubit go in sequence — so minimize the number of
> `fswap`-*layers* (with the 2-CNOT `fswap`, `depth_2q` = twice the layer count).
> The lazy route does the three transpositions one at a time (reuse Problem 2);
> you can do better by *interleaving* moves from different transpositions into
> shared layers.

> 📦 **Deliverables**
>
> (a) Your circuit.  (b) Its two-qubit depth.


In [ ]:
# YOUR CODE HERE.  Concatenate fswap(j) lists, e.g.  fswap(4) + fswap(3) + ...
my_perm = []

print("verify_fp:", verify_fp(my_perm, [(1, 5), (2, 6), (3, 7)]))
print("depth_2q :", depth_2q(my_perm))


Inspect the result — after the permutation, modes 1 and 5 should have swapped
(the leaf $\gamma_2 / \gamma_3$ now carries the JW weight that $\gamma_{10} /
\gamma_{11}$ used to, and vice versa), and similarly for the other pairs.  The
fixed modes 0, 4, 8 are unchanged:


In [ ]:
fig, ax = draw_majorana_tree(my_perm,
                             title="After the (1 5)(2 6)(3 7) reflection")
plt.show()


---

## Problem 4 — One Trotter step of the spinless Fermi–Hubbard model

*Applying the permutation on real workload*

The 9 modes are the sites of a 3×3 spatial lattice — **the same grid as the
snake JW layout**, so mode $i$ sits at JW position $i$.  The green figure below
draws this lattice with its 12 nearest-neighbour bonds.  The whole problem
turns on the contrast between two kinds of bond:

* **horizontal bonds** run *along* the snake → their modes are **JW-adjacent**
  ($d = 1$, no parity string) — **cheap**;
* **vertical bonds** (highlighted) jump *between* folded rows, so most have
  **JW-far** endpoints ($d = 3$ or $5$) with a long parity string — **expensive**;
  two of them, $(5,6)$ and $(2,3)$, happen to land **JW-adjacent** ($d = 1$, cheap).


In [ ]:
fig, _ = snake_and_lattice_figure(figsize=(11, 5.0))
plt.show()


Reading the bonds off the figure, the spinless Fermi–Hubbard **hopping**
Hamiltonian $H = -t \sum_{\langle i,j\rangle} (a_i^\dagger a_j + a_j^\dagger
a_i)$ written out explicitly over those 12 bonds is

$$
\begin{aligned}
H = -t \big[\;
&\underbrace{h_{01} + h_{12} + h_{34} + h_{45} + h_{67} + h_{78}}_{\text{6 horizontal bonds (cheap)}} \\
+\;&\underbrace{h_{05} + h_{56} + h_{14} + h_{47} + h_{23} + h_{38}}_{\text{6 vertical bonds (mostly expensive)}}
\;\big],
\qquad h_{ij} \equiv a_i^\dagger a_j + a_j^\dagger a_i .
\end{aligned}
$$

One first-order Trotter step at angle $\theta = t\,\Delta t$ is the ordered
product $U(\theta) = \prod_{\langle i,j\rangle} \exp(i\,\theta\,h_{ij})$ —
horizontal bonds first, then vertical, in the order above.  The lists
`HORIZONTAL_EDGES`, `VERTICAL_EDGES`, `HOPPING_EDGES` hold that exact ordering
(the same one the checker uses).

> 🎯 **Task**
>
> 1. **Translate each hopping term to Paulis.** Use `hopping_pauli_strings(i, j)`, or read off $a_i^\dagger a_j + \text{h.c.} = \tfrac{1}{2}(X_p X_q + Y_p Y_q)\,Z_{p+1}\cdots Z_{q-1}$ for $p < q$.
> 2. **Build $U(\theta)$ out of native gates.** Each hopping term is a multi-qubit Pauli rotation $\exp(i\tfrac{\theta}{2}P)$. The simplest way is `qml.PauliRot(-theta, "XZ...ZX", wires=...)` — it lowers to a native CNOT staircase (or roll your own). Use your `fswap` for any fermionic permutation (including your Problem 3 circuit), and keep every two-qubit gate on a **grid-adjacent** pair.
> 3. **Measure.** `depth_2q(circuit, "2dnn")` checks the gate set + adjacency and returns the two-qubit depth — your score.
>
> Confirm correctness with `verify_trotter_step(circuit, theta)`, then minimize the depth.

> 💡 **Hint**
>
> Compare the Pauli strings of horizontal vs vertical hops under the snake. Is
> there a fermionic permutation — perhaps one you already built — that you could
> apply *partway through* the step to turn the expensive family into the cheap one?


### Native gate set, topology, and how depth is measured

There is **no built-in compiler** in this pset. `depth_2q` only **checks** it and reports the two-qubit depth.

**Native gate set:** single-qubit gates (anything — $H, S, S^\dagger, R_Z$, …
are free) plus the two-qubit **CNOT (=CX)**, **CY**, and **CZ**.  Any other gate
is accepted *iff* it `decomposition()`s down to those — so `qml.SWAP` counts as 3
CNOTs, your `fswap` as whatever you built it from, and **`qml.PauliRot` as its
CNOT staircase**.  (We do *not* do Clifford+$T$; $R_Z(\theta)$ stays continuous.)

**Topology:** on `"2dnn"`, every two-qubit gate must act on a *grid-adjacent*
pair (the snake makes JW-adjacent = grid-adjacent); `depth_2q(circuit, "2dnn")`
**raises** otherwise.  On `"ata"` any pair is allowed.

**A hopping term is a multi-qubit Pauli rotation** $\exp(i\tfrac{\theta}{2}P)$.
Two ways to gather its parity:

| gather network | how to get it | 2-qubit depth | legal on |
|---|---|---|---|
| **linear CNOT staircase** | `qml.PauliRot` gives this for free | $\sim 2(w{-}1)$ | 2D-NN *(contiguous support)* and all-to-all |
| **balanced binary CNOT tree** | build it yourself (long-range CNOTs) | $\sim 2\lceil\log_2 w\rceil$ | **all-to-all only** |

For Problem 4 (2D-NN) the staircase from `qml.PauliRot` is exactly what you
want.  That last row is the whole story of Problem 5: the shallower tree needs
long-range CNOTs, so it's rejected on `"2dnn"` but shines on `"ata"` — and since
`qml.PauliRot` only emits the staircase, you build the tree by hand.


First, feel the cost gap — read off the Pauli weight of a horizontal vs a vertical hop:


In [ ]:
theta = 0.137  # one Trotter angle, used throughout Problems 4-5

# Pauli forms of representative hops (read the edges off the green figure).
for label, (i, j) in (("horizontal (0,1)", (0, 1)),
                      ("vertical   (0,5)", (0, 5))):
    xx, yy = hopping_pauli_strings(i, j)
    print(f"  {label}:  XX = {xx.to_label()}   YY = {yy.to_label()}")
# A horizontal hop is weight-2 (one CNOT layer); a vertical hop drags a long
# Z-string -> a deep staircase.  A naive in-place step (all 12 hops, chain
# rotations) costs 2D-NN depth 70 -- that is the baseline to beat.


Now build your Trotter step.

> 💡 **Hint**
>
> *Strategy (ignore it and try your own first).* Sandwich the vertical-edge
> rotations between two copies of the Problem-3 diagonal reflection: under the
> reflected order every vertical pair is JW-adjacent, so each vertical rotation is
> a short adjacent staircase, and the two reflections cancel — the unitary is
> unchanged.


In [ ]:
# YOUR CODE HERE — assemble U(theta) from native gates.
# (qml.PauliRot(-theta, "...", wires=...) for each hop; your fswap for permutations.)
my_trotter_step = [
    # ...
]

# Once you have something, run:
# print("verify_trotter_step:", verify_trotter_step(my_trotter_step, theta))
# print("2D-NN depth :", depth_2q(my_trotter_step, "2dnn"))


> 📦 **Deliverables**
>
> (a) Your circuit.  (b) Its 2D-NN two-qubit depth.  (c) A short markdown cell
> explaining your strategy and why it works.


---

## Problem 5 — The same Trotter step on all-to-all hardware

*Re-target the Problem-4 workload to all-to-all hardware, and understand the impacts of connectivity constraints.*

Repeat Problem 4 — one Trotter step of the same Hamiltonian on the same 9
modes — but now target **all-to-all** hardware.  Only the scoring changes:
`depth_2q(circuit, "ata")` drops the grid-adjacency requirement, so your
two-qubit gates may act on **any** pair.  In particular you may now gather a
rotation's parity with the **balanced binary CNOT tree** (long-range CNOTs)
that was *rejected* on 2D-NN.  `verify_trotter_step` is unchanged.

> 🎯 **Task**
>
> Give the lowest `depth_2q(circuit, "ata")` you can, and confirm correctness with
> `verify_trotter_step`. Two routes — pick either:
>
> - **Adapt Problems 1–4.** Your swap/permutation circuits still work; now you may also rebuild each hopping rotation with a log-depth **tree** parity network.
> - **Start fresh.** All-to-all may admit a structurally different circuit than anything natural on the grid.

> 💡 **Hint**
>
> Two moves help, but only one is *about* all-to-all. (i) A hop's `XX` and `YY`
> rotations are a single **Givens rotation** `qml.IsingXY(2*theta, [p, q])` (depth
> 2) — handy, but legal on *any* topology. (ii) All-to-all lets distant qubits
> interact directly, yet it does **not** erase the JW parity strings: a JW-far hop
> is still a high-weight rotation. What all-to-all uniquely makes cheaper is the
> parity *gather* — a balanced **tree** instead of a **staircase**. Use (i)
> everywhere, then ask which gather your hops should use.

> 📦 **Deliverables**
>
> (a) Your circuit.  (b) Its `depth_2q(circuit, "ata")`.  (c) A short markdown
> cell on what your all-to-all circuit does differently from your Problem-4
> circuit, and why.


In [ ]:
# YOUR CODE HERE.
my_trotter_ata = [
    # ...
]

# Once you have something, run:
# print("verify_trotter_step:", verify_trotter_step(my_trotter_ata, theta))
# print(" all-to-all depth :", depth_2q(my_trotter_ata, "ata"))


---

### 🏁 Done!

We've built the fermionic-simulation workflow into code: we
built the FSWAP gate from first principles, routed fermions with bubble walks,
and scheduled a multi-transposition permutation for minimum depth — the
fermionic-swap-network toolkit that is a genuine workhorse of real Hamiltonian
simulation. Then we turned it loose on an actual Trotter step; and however you
drove the depth down — routing the modes together, folding a hop into a local
Givens and gathering its parity in place, or trading a staircase for a log-depth
tree — those are exactly the moves a fermionic-simulation compiler makes.

Peek inside `pset_tools/` to see how the checker, the Majorana visualizer, and
the depth counter work — they are short, readable, and worth knowing for your
own work with Jordan–Wigner.  (On Colab it's the `rqs2026-pset/` folder in the
file pane on the left; locally it sits next to this notebook.)
